### Name: Qian Ling GitHub 
### Username: qianling-sketch
### USC ID: 7728910687

## 1. Time Series Classification Part 1

In [29]:
import os
import pandas as pd
import numpy as np

data_path = "data/AReM"

print(os.listdir(data_path))

['bendingType.pdf', '.DS_Store', 'bending1', 'walking', 'sensorsPlacement.pdf', 'bending2', 'standing', 'sitting', 'lying', 'cycling']


### (b)

In [30]:
activities = [
    "bending1", "bending2", "cycling",
    "lying", "sitting", "standing", "walking"
]

train_files = []
test_files = []

for activity in activities:
    folder = os.path.join(data_path, activity)
    
    for file in os.listdir(folder):
        if not file.endswith(".csv"):
            continue
        
        file_num = int(file.replace("dataset", "").replace(".csv", ""))
        file_path = os.path.join(folder, file)
        
        if activity in ["bending1", "bending2"]:
            if file_num in [1, 2]:
                test_files.append(file_path)
            else:
                train_files.append(file_path)
        else:
            if file_num in [1, 2, 3]:
                test_files.append(file_path)
            else:
                train_files.append(file_path)

print("Training instances:", len(train_files))
print("Test instances:", len(test_files))
print("Total instances:", len(train_files) + len(test_files))

Training instances: 69
Test instances: 19
Total instances: 88


### (c)(i)

The commonly used ones include: minimum value, maximum value, mean value, median, standard deviation, variance, first/third quartiles, range, interquartile range IQR, etc.

### (ii)

In [12]:
cols = ['time', 'avg_rss12', 'var_rss12', 'avg_rss13',
        'var_rss13', 'avg_rss23', 'var_rss23']

def extract_features(path, label):
    df = pd.read_csv(path, comment='#', header=None,
                 usecols=range(7), names=cols,
                 sep=r'[,\s]+', engine='python',
                 on_bad_lines='skip')
    df = df.dropna()
    df = df.drop(columns='time')

    feats = {}
    for i, c in enumerate(df.columns, start=1):
        x = df[c].values
        feats[f'min{i}']       = np.min(x)
        feats[f'max{i}']       = np.max(x)
        feats[f'mean{i}']      = np.mean(x)
        feats[f'median{i}']    = np.median(x)
        feats[f'std{i}']       = np.std(x, ddof=1)
        feats[f'1st_quart{i}'] = np.percentile(x, 25)
        feats[f'3rd_quart{i}'] = np.percentile(x, 75)
    feats['activity'] = label
    feats['n_rows'] = len(df)            
    return feats

train_df = pd.DataFrame([extract_features(p, p.split(os.sep)[-2]) for p in train_files])
test_df  = pd.DataFrame([extract_features(p, p.split(os.sep)[-2]) for p in test_files])

# Merge into the 88-row table in the title, with Instance ranging from 1 to 8
all_df = pd.concat([train_df, test_df], ignore_index=True)
all_df.index = range(1, len(all_df) + 1)
all_df.index.name = 'Instance'

print(train_df.shape, test_df.shape, all_df.shape)
print(all_df['n_rows'].unique())         
all_df.head()

(69, 44) (19, 44) (88, 44)
[480 479]


,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,...,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6,activity,n_rows
Instance,,,,,,,,,,,,,,,,,,,,,
1,36.25,48.00,43.969125,44.50,1.618364,43.31,44.67,0.0,1.50,0.413125,...,23.75,0.0,2.96,0.555312,0.49,0.487826,0.00,0.83,bending1,480
2,37.00,48.00,43.454958,43.25,1.386098,42.50,45.00,0.0,1.58,0.378083,...,24.00,0.0,5.26,0.679646,0.50,0.622534,0.43,0.87,bending1,480
3,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,...,36.33,0.0,2.18,0.613521,0.50,0.524317,0.00,1.00,bending1,480
4,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,...,31.25,0.0,1.79,0.383292,0.43,0.389164,0.00,0.50,bending1,480
5,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,...,36.50,0.0,1.79,0.493292,0.43,0.513506,0.00,0.94,bending1,480


In [16]:
for p in train_files + test_files:
    features = extract_features(p, p.split(os.sep)[-2])
    if features['n_rows'] != 480:
        print(p, features['n_rows'])

AReM/sitting/dataset8.csv 479


One malformed row in `sitting/dataset8.csv` was skipped during data cleaning, resulting in 479 valid observations for this instance.

### (iii)

In [17]:
feature_cols = [c for c in all_df.columns if c not in ['activity', 'n_rows']]
X = all_df[feature_cols].values        # 88 x 42

rng = np.random.default_rng(0)
B = 2000
n = X.shape[0]

boot_std = np.empty((B, X.shape[1]))
for b in range(B):
    idx = rng.integers(0, n, n)       
    boot_std[b] = X[idx].std(axis=0, ddof=1)

std_table = pd.DataFrame({
    'std':    X.std(axis=0, ddof=1),
    'CI_5%':  np.percentile(boot_std, 5,  axis=0),
    'CI_95%': np.percentile(boot_std, 95, axis=0)
}, index=feature_cols)
std_table

,std,CI_5%,CI_95%
min1,9.569975,8.255488,10.798825
max1,4.394362,3.356895,5.275593
mean1,5.335718,4.696066,5.883307
median1,5.440054,4.786646,5.987759
std1,1.772153,1.579102,1.952310
1st_quart1,6.153590,5.587456,6.653489
3rd_quart1,5.138925,4.300110,5.826979
min2,0.000000,0.000000,0.000000
max2,5.062729,4.647263,5.418592
mean2,1.574164,1.405500,1.703020


The standard deviation of each of the 42 time-domain features was estimated across the 88 instances. A 90% bootstrap confidence interval was constructed using the 5th and 95th percentiles of the bootstrap distribution.

### (iv)

from sklearn.feature_selection import f_classif

nonconstant_cols = [
    c for c in feature_cols
    if all_df[c].nunique() > 1
]

F, p = f_classif(
    all_df[nonconstant_cols],
    all_df['activity']
)

rank = pd.Series(F, index=nonconstant_cols).sort_values(ascending=False)
rank.head(10)

Based on the ANOVA F-scores, I selected **mean, median, and third quartile** as the three most important time-domain features. These features consistently showed high discriminative power across the six time series.

## 2

### (a)

The cubic regression will have a training RSS that is less than or equal to that of the linear regression. The cubic model is more flexible and contains the linear model as a special case when the coefficients of X^2 and X^3 are zero. Therefore, it can fit the training data at least as well as the linear model.

### (b)

The linear regression is expected to have a lower test RSS. Since the true relationship is linear, the additional quadratic and cubic terms are unnecessary and may lead to overfitting. Therefore, the cubic model may have higher variance and perform worse on unseen data.

### (c)

The cubic regression will have a training RSS that is less than or equal to that of the linear regression. Since the cubic model is more flexible, it can fit the training observations at least as well as the linear model.

### (d)

There is not enough information to determine, because it depends on how far the real relationship deviates from linearity. If the true relationship is close to linear, the linear model may perform better. If the relationship is strongly nonlinear and can be approximated well by a cubic function, the cubic model may have a lower test RSS.